Solo se proporcionan las funciones que se deben realizar, mas no todo el código ni los headers que estos utilizan.

**2.1 Bisección**

In [ ]:
/// FUNCION QUE IMPLEMENTA EL METODO DE LA BISECCION (VERSION MAS CONCISA)
real mn_biseccion (
real (*f)(const real), /// función a la cual se calcula un cero
real &a, real &b, /// intervalo inicial para buscar la raíz
const real TOL,  /// tolerancia para parar las iteraciones del algoritmo
int &Niter) /// número de iteraciones realizadas por el método
            ///       Si Niter=-1 la función ha terminado mal
{
  if (f(a) * f(b) > 0) { Niter = -1; return -1.; }
  Niter = 0;
  while (mn_distancia(a, b) >= TOL) {
    real c = (a + b) / 2;
    if (f(c) == 0.) return c;
    if (f(a) * f(c) < 0) b = c;
    else a = c;
    Niter++;
  }
  return (a + b) / 2;
}

**2.2 Regula Falsi**

In [ ]:
/// FUNCION QUE IMPLEMENTA EL METODO DE LA REGULA FALSI (VERSION MAS CONCISA)
int mn_regula_falsi (
real (*f)(real), /// función a la cual se calcula un cero
real &a, real &b, /// intervalo inicial para buscar la raíz
real &x, /// valor de salida de la raíz
real TOL,  /// tolerancia para parar las iteraciones del algoritmo
int NiterMax) /// número máximo de iteraciones permitidas
{
    if (f(a) * f(b) >= 0.) return -1;
    for (int i = 0; i < NiterMax; i++) {
        x = a - (b - a) / (f(b) - f(a)) * f(a);
        if (f(x) == 0. || mn_distancia(a, b) < TOL) return i;
        (f(a) * f(x) < 0 ? b : a) = x;
    }
    return -1;
}

**2.3 Newton-Raphson**

In [ ]:
/// FUNCION QUE IMPLEMENTA EL METODO DE NEWTON-RAPHSON APROXIMANDO LA FUNCION DERIVADA
/// LA FUNCIÓN DEVUELVE EL NÚMERO DE ITERACIONES REALIZADAS SI TERMINA BIEN Y DEVUELVE -1
/// EN CASO CONTRARIO
int mn_newton_raphson (
real (*f)(real), /// funcion sobre la que se calcula el cero
real &x0, /// raíz inicial que actualiza la función
int NiterMax, /// número de iteraciones máximo
real TOL) /// tolerancia para parar el algoritmo
{
    for (int i = 0; i < NiterMax; i++) {
        if (f(x0) == 0) return i;
        real df = mn_derivada1(f, x0);
        if (df == 0) return i;
        real x1 = x0 - f(x0) / df;
        if (mn_distancia(x0, x1) <= TOL) { x0 = x1; return i; }
        x0 = x1;
    }
    return -1;
}

**2.4 Secante**

In [ ]:
/// FUNCION QUE IMPLEMENTA EL METODO DE LA SECANTE (VERSION MAS CONCISA)
int mn_secante (
real (*f)(real), /// funcion sobre la que se calcula el cero
real &x0, /// primera aproximación raíz que actualiza la función
real &x1, /// segunda aproximación raíz que actualiza la función
int NiterMax, /// número de iteraciones máximo
real TOL) /// tolerancia para parar el algoritmo
{
    for (int i = 0; i < NiterMax; i++) {
        if (f(x1) == 0.0) return i;
        real df = (f(x1) - f(x0)) / (x1 - x0);
        if (df == 0.0) return -1;
        real x2 = x1 - f(x1) / df;
        if (mn_distancia(x1, x2) <= TOL) { x0 = x1; x1 = x2; return i; }
        x0 = x1; x1 = x2;
    }
    return -1;
}

**2.5 Muller**

In [ ]:
int mn_muller (
real (*f)( real), /// funcion sobre la que se calcula el cero
real &x0, /// raíz inicial que actualiza la función
int NiterMax, /// número de iteraciones máximo
real TOL) /// tolerancia para parar el algoritmo
{
  /// HACER ALUMNO
  for (int k=0;k<NiterMax;k++){
    if (f(x0)==0) return(k);
    real derivada1 = mn_derivada1(f, x0);
    real derivada2 = mn_derivada2(f, x0);
    real aux = derivada1*derivada1-2*f(x0)*derivada2;
    if (aux<0) return(-1);
    real raiz = sqrt(aux);
    real x1=x0 + (-derivada1+raiz)/derivada2;
    real x2=x0 + (-derivada1-raiz)/derivada2;
    real xant=x0;
    if (fabs(x0-x1) < fabs(x0-x2)){
        x0=x1;
    } else {
        x0=x2;
    }
    if(mn_distancia(xant, x0)<TOL) return(k);
  }
  return(-1);
}

**2.6 Polinomios Horner**

In [ ]:
void mn_evaluar_polinomio_horner(
Array1D< real > &a /** coeficientes polinomio */,
real x /** valor donde se evalua el polinomio */,
real &Px /** evaluación del polinomio en x*/,
real &PPx /** evaluación de la derivada del polinomio en x*/){
  /// HACER ALUMNO
	Px = 0; PPx = 0;
	for (int k = a.dim() - 1; k >= 0; --k) {
		PPx = PPx * x + Px;
		Px  = Px  * x + a[k];
	}
}

Array1D< real > mn_calcular_derivada_polinomio(
Array1D< real > &a /** coeficientes del polinomio */){
  /// HACER ALUMNO
	Array1D<real> b(a.dim() - 1);
	for (int k = 0; k < b.dim(); ++k)
		b[k] = a[k + 1] * (k + 1);
	return b;
}

**2.7 Ceros Polinomios**

In [ ]:
Array1D<real> calcular_derivada_polinomio(Array1D<real> &a) {
	int N = a.dim();
	Array1D<real> b(N>1 ? N-1 : 0);
	for (int i = 0; i < N-1; ++i) b[i] = (i+1) * a[i+1];
	return b;
}

Array1D<real> ceros_polinomio_desde_ceros_derivada(
	Array1D<real> &a, Array1D<real> &d, real TOL)
{
	int N = a.dim(), Nd = d.dim();
	if (N>0 && a[N-1] == 0.) return d;
	real maximo = 0;
	for (int i = 0; i < N-1; ++i) maximo = mn_max(maximo, mn_abs(a[i]));
	real Pmax = 1 + maximo/mn_abs(a[N-1]);
	Array1D<real> inter;
	inter.push_back(-Pmax);
	for (int i = 0; i < Nd; ++i) inter.push_back(d[i]);
	inter.push_back(Pmax);

	Array1D<real> ceros;
	for (int i = 0; i+1 < inter.dim(); ++i) {
		real fa = evaluar_polinomio(a, inter[i]),
			 fb = evaluar_polinomio(a, inter[i+1]);
		if (fa*fb <= 0)
			ceros.push_back(calculo_cero_en_intervalo(
				a, inter[i], inter[i+1], TOL));
	}
	return ceros;
}

Array1D<real> ceros_polinomio(Array1D<real> &a, real TOL) {
	int N = a.dim();
	Array1D<real> ceros;
	for (int i = N-2; i >= 0; --i) {
		Array1D<real> poly = a;
		for (int j = 0; j < i; ++j) poly = calcular_derivada_polinomio(poly);
		ceros = ceros_polinomio_desde_ceros_derivada(poly, ceros, TOL);
	}
	return ceros;
}